# MedMNIST analysis
Vérification rapide : 2 exemples (224x224 natif) par dataset.
On lit seulement les 2 premières images de chaque .npz (sans tout décompresser).

In [ ]:
import os
import zipfile
import numpy as np
from numpy.lib import format as npformat
import matplotlib.pyplot as plt

ROOT = os.path.expanduser("~/.medmnist")
SIZE = 224
N_PER = 2

STAGE_1 = ["pathmnist", "dermamnist", "pneumoniamnist", "bloodmnist", "organcmnist", "organsmnist"]
STAGE_2 = ["chestmnist", "octmnist", "retinamnist", "breastmnist", "tissuemnist", "organamnist"]
DATASETS = STAGE_1 + STAGE_2


def load_first_images(name, split="train", n=2):
    path = os.path.join(ROOT, f"{name}_{SIZE}.npz")
    if not os.path.exists(path):
        path = os.path.join(ROOT, f"{name}.npz")
    with zipfile.ZipFile(path) as z:
        with z.open(f"{split}_images.npy") as f:
            version = npformat.read_magic(f)
            shape, _, dtype = npformat._read_array_header(f, version)
            per_img = int(np.prod(shape[1:]))
            buf = f.read(n * per_img * dtype.itemsize)
            imgs = np.frombuffer(buf, dtype=dtype).reshape((n,) + shape[1:])
    return imgs, shape

In [ ]:
fig, axes = plt.subplots(len(DATASETS), N_PER, figsize=(N_PER * 2.5, len(DATASETS) * 2.5))

for row, name in enumerate(DATASETS):
    try:
        imgs, shape = load_first_images(name, split="train", n=N_PER)
    except Exception as e:
        print(f"skip {name}: {e}")
        for col in range(N_PER):
            axes[row, col].axis("off")
        continue

    for col in range(N_PER):
        ax = axes[row, col]
        ax.imshow(imgs[col], cmap="gray")
        ax.axis("off")
        if col == 0:
            ax.set_title(f"{name}  {tuple(shape[1:])}  n={shape[0]}", loc="left", fontsize=9)

plt.tight_layout()
plt.show()